# Первый ML-эксперимент своими руками

Сегодня соберём короткий ML-эксперимент: поймём таблицу, отделим train от test, построим простые решения и один раз проверим их на новых объектах. Notebook — рабочий инструмент занятия; его не нужно сдавать.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

## Знакомый churn case

Вопрос перед кодом: что является объектом в этой задаче? Что предсказываем, в какой момент и на каком горизонте?

In [ ]:
demo_df = pd.read_csv("data/demo-churn.csv")
demo_df

Каждая строка — аккаунт в момент `t0`. Target `churn_30d` равен `1`, если подписка будет отменена в следующие 30 дней, и `0` иначе.

У нас есть два столбца про активность: `active_days_14d` и `active_days_next_7d`. Какой из них можно использовать для прогноза в момент `t0`?

## Быстро вспоминаем DataFrame

Посмотрим на несколько обычных команд. Каждая отвечает на свой простой вопрос о таблице.

In [ ]:
demo_df.head(3)

In [ ]:
demo_df.shape

In [ ]:
demo_df.dtypes

## Делим demo на train и test

Сегодня используем случайное разбиение 80/20 как учебный вариант. В реальной задаче способ разбиения выбирают так, чтобы он был похож на будущее применение модели.

In [ ]:
demo_features = ["tenure_months", "active_days_14d"]
demo_X = demo_df[demo_features]
demo_y = demo_df["churn_30d"]

demo_X_train, demo_X_test, demo_y_train, demo_y_test = train_test_split(
    demo_X, demo_y, test_size=0.20, random_state=2026
)

demo_X_train.shape, demo_X_test.shape

Какой класс в train является самым частым? Какую accuracy постоянный прогноз имеет **на train**?

In [ ]:
demo_y_train.value_counts()

## Пишем MostFrequentClassifier с нуля

Что модели нужно запомнить? Когда она узнаёт это значение? Что должен делать `fit`? Что использует `predict`? Нужны ли `predict` правильные ответы новых объектов?

In [ ]:
class MostFrequentClassifier:
    pass

In [ ]:
model = MostFrequentClassifier()
vars(model)

После появления `fit` обучим модель на train. Что изменится в `vars(model)`?

In [ ]:
# model.fit(demo_X_train, demo_y_train)
# vars(model)

Теперь добавим `predict`: он получает новые признаки и возвращает сохранённый самый частый класс нужное число раз.

In [ ]:
# demo_predictions = model.predict(demo_X_test)
# demo_predictions

## Проверяем ручной baseline

После прогноза можно сравнить его с известными ответами test. Это уже оценка на объектах, которые не участвовали в `fit`.

In [ ]:
# manual_accuracy = accuracy_score(demo_y_test, demo_predictions)
# manual_accuracy

## Тот же baseline в scikit-learn

`DummyClassifier(strategy="most_frequent")` реализует ту же идею. Он принимает признаки и строки target так же, как другие sklearn-модели.

In [ ]:
demo_dummy = DummyClassifier(strategy="most_frequent")
demo_dummy.fit(demo_X_train, demo_y_train)
demo_dummy_predictions = demo_dummy.predict(demo_X_test)

accuracy_score(demo_y_test, demo_dummy_predictions), demo_dummy_predictions == demo_predictions

# Переходим к новому набору

Теперь применим тот же ход рассуждений к реальным данным.

## Bank Marketing

**Источник:** S. Moro, P. Cortez, P. Rita, *Bank Marketing*,
UCI Machine Learning Repository.
DOI: https://doi.org/10.24432/C5K306. Лицензия: CC BY 4.0.

Данные относятся к телефонным маркетинговым кампаниям португальского банка.
Банк связывался с клиентами по телефону и предлагал оформить **срочный банковский вклад**
(*term deposit*). Для одного клиента в рамках кампании могло потребоваться
несколько контактов.

Результат кампании для клиента — оформил он предложенный вклад или нет.


In [ ]:
bank_df = pd.read_csv("data/bank-week01.csv")
bank_df

### Постановка

**Задача.** Непосредственно перед очередным телефонным контактом предсказать,
оформит ли клиент срочный банковский вклад по итогам текущей маркетинговой кампании.

**Строка.** Запись о клиенте в контексте маркетинговой кампании. Стабильного
`customer_id` в наборе нет, поэтому мы не предполагаем, что каждая строка
соответствует уникальному человеку.

**Target.** `y ∈ {yes, no}` — был ли оформлен срочный банковский вклад.

**Момент прогноза.** Непосредственно перед текущим телефонным контактом.

**Горизонт.** До результата текущей кампании; точный календарный горизонт
описание набора не задаёт.

In [ ]:
bank_df.shape

In [ ]:
bank_df.drop(columns="y").head(3)

In [ ]:
bank_df.dtypes

### Краткий словарь столбцов

| Column | Смысл | Важное кодирование |
|---|---|---|
| `age` | возраст клиента | число лет |
| `job` | тип занятости | `unknown` — категория источника |
| `education` | уровень образования | `unknown`, `primary`, `secondary`, `tertiary` |
| `balance` | средний годовой баланс | евро; число может быть отрицательным |
| `housing` | есть ли жилищный кредит | `yes` / `no` |
| `loan` | есть ли личный кредит | `yes` / `no` |
| `pdays` | дней с последнего контакта в предыдущей кампании | `-1`: прежде не связывались |
| `previous` | число контактов до текущей кампании | неотрицательное целое |
| `poutcome` | исход предыдущей кампании | `unknown`, `other`, `failure`, `success` |
| `duration` | длительность текущего телефонного разговора | секунды |
| `y` | подписка на срочный депозит | `yes` / `no` |

Вопрос: при prediction moment непосредственно перед звонком можно ли использовать `duration`? Почему?

## Выбираем допустимые признаки

`duration` существует в историческом CSV, но становится известен после разговора. Для выбранного момента прогноза это утечка из будущего, поэтому в `X_allowed` его не включаем.

In [ ]:
X_allowed = [
    "age", "job", "education", "balance", "housing", "loan",
    "pdays", "previous", "poutcome",
]

X = bank_df[X_allowed]
y = bank_df["y"]

## Делим Bank Marketing на train и test

До финальной проверки будем исследовать данные и выбирать правило только по `X_train` и `y_train`. `y_test` сохраняем для конца эксперимента.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=2026
)

X_train.shape, X_test.shape

## Как распределён target в train?

Какой класс окажется самым частым? Какую accuracy примерно даст модель, которая всегда говорит `no`?

In [ ]:
# Вызовите value_counts() для y_train.
class_counts = None

In [ ]:
# Вызовите value_counts(normalize=True) для y_train.
class_proportions = None

In [ ]:
# Эту ячейку можно просто выполнить, она построит график распределения классов в train.
target_counts = y_train.value_counts()
ax = target_counts.plot.bar(color=["#2563eb", "#f59e0b"], rot=0)
ax.set_xlabel("Ответ y")
ax.set_ylabel("Число объектов в train")
ax.set_title("Распределение target в train")
for bar, count in zip(ax.patches, target_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, count + 35, str(count),
            ha="center", va="bottom")
plt.tight_layout()

## Специальные значения в данных

Отсутствие информации в таблице не всегда записано как `NaN`. Посмотрите на результат: какие специальные значения здесь есть? Не нужно ничего заменять или кодировать.

In [ ]:
# Метод eq() возвращает DataFrame, где True для ячеек, равных "unknown", и False для остальных. 
# Видим что в столбцах job, education, housing, loan, poutcome есть пропуски, закодированные в виде строки "unknown".
X_train[
    ["job", "education", "housing", "loan", "poutcome"]
].eq("unknown").sum() 

In [ ]:
(X_train["pdays"] == -1).sum(), (X_train["poutcome"] == "unknown").sum()

В словаре `pdays = -1` означает, что с клиентом не связывались в предыдущей кампании; `unknown` — отдельная категория источника. Видим что числа совпадают, что логично: не было связей с клиентом - нет и предыдущего результата.

## Что train подсказывает для простого правила?

Посмотрим, как доля `yes` меняется между исходами предыдущей кампании.

In [ ]:
# Функция pd.crosstab() строит таблицу сопряженности (contingency table) для двух категориальных переменных. 
# В ячейках таблицы будет число (доля) объектов, соответствующих каждой комбинации значений переменных.
pd.crosstab(
    X_train["poutcome"],
    y_train,
    normalize="index",
).round(3)



Видим, что в категории `success` доля `yes` заметно выше. Если в прошлый раз клиент был согласен, то скорее будет склонен согласиться и в этот раз. Зафиксируем эвристику:

`poutcome == "success"` → `"yes"`  
иначе → `"no"`

In [ ]:
predictions = []

for result in X_test["poutcome"]:
    if result == "success":
        predictions.append("yes")
    else:
        predictions.append("no")

# Получили предсказания на основе простого правила. В принципе, и это можно было обернуть в интерфейс класса с fit/predict.
y_pred_rule = predictions

## Строим majority baseline

MostFrequentClassifier уже показал, что делают `fit` и `predict`. Теперь используем библиотечный `DummyClassifier` для того же baseline.

In [ ]:
# Создайте DummyClassifier с strategy="most_frequent".
dummy = None

In [ ]:
# Обучите dummy на X_train и y_train.
# dummy.fit(X_train, y_train)

In [ ]:
# Получите прогнозы dummy для X_test.
y_pred_dummy = None

## Теперь открываем test

У нас готовы два зафиксированных решения: наиболее частый класс и простое доменное правило. Посмотрим на их результаты на test.

In [ ]:
dummy_accuracy = accuracy_score(y_test, y_pred_dummy)
dummy_accuracy

In [ ]:
rule_accuracy = accuracy_score(y_test, y_pred_rule)
rule_accuracy

In [ ]:
100 * (rule_accuracy - dummy_accuracy)

## Вопросы к результату

- Какой класс оказался наиболее частым в `y_train` и какова его доля?
- Какую accuracy получил majority baseline?
- Выиграло ли правило по `poutcome`? На сколько процентных пунктов?
- Для чего мы использовали train, а для чего test?

## Коротко

прикладная задача → prediction moment → допустимые признаки → train/test → исследование train → baseline → простое правило → test → аккуратный вывод.